# Stage 1 $-$ Frozen conventions and canonical Hamiltonian core

- Gabriel Wendell Celestino Rocha
---

### **Here we demonstrates the tested public API.**

## Objectives

1. locate the project-wide physical and numerical conventions;
2. construct compatible open, ribbon, and magnetic-Bloch Harper-Hofstadter Hamiltonians;
3. verify Hermiticity rather than assuming it;
4. understand why analytic Bloch derivatives are prepared before Kubo calculations.

In [1]:
import numpy as np

from qhe.conventions import CONVENTIONS
from qhe.models import (
    HarperHofstadterParameters,
    bloch_hamiltonian,
    bloch_hamiltonian_derivatives,
    magnetic_brillouin_zone,
    open_hamiltonian,
    ribbon_hamiltonian,
)
from qhe.validation import hermiticity_residual

## Frozen Convention Summary

The complete record is in `docs/CONVENTIONS.md`

In [2]:
print("> Gauge:", CONVENTIONS.gauge)
print("> Electron charge:", CONVENTIONS.electron_charge)
print("> Natural hbar:", CONVENTIONS.hbar)
print("> Site indexing:", CONVENTIONS.site_indexing)
print("> FHS orientation for Stage 2:", CONVENTIONS.fhs_orientation)

> Gauge: Landau gauge A = (0, Bx, 0)
> Electron charge: -1.0
> Natural hbar: 1.0
> Site indexing: index(m, n; Ly) = m * Ly + n; reshape -> (Lx, Ly)
> FHS orientation for Stage 2: Use -Arg[U_x(k) U_y(k+x) U_x(k+y)^(-1) U_y(k)^(-1)] so that the discrete result matches the Berry convention above.


## Build the three compatible Harper-Hofstadter geometries

All are derived from the same Landau-gauge Peierls Hamiltonian for $\phi=1/3$.

In [3]:
params = HarperHofstadterParameters(p=1, q=3, tx=1.0, ty=1.0)
zone = magnetic_brillouin_zone(params)

h_bulk = bloch_hamiltonian(kx=0.12, ky=-0.31, parameters=params)
h_ribbon = ribbon_hamiltonian(lx=12, ky=-0.31, parameters=params)
h_open = open_hamiltonian(lx=6, ly=8, parameters=params)

print("> Magnetic BZ:", zone)
print("> Bulk shape:", h_bulk.shape)
print("> Ribbon shape:", h_ribbon.shape)
print("> Open shape:", h_open.shape)

> Magnetic BZ: MagneticBrillouinZone(kx_min=-1.0471975511965976, kx_max=1.0471975511965976, ky_min=-3.141592653589793, ky_max=3.141592653589793)
> Bulk shape: (3, 3)
> Ribbon shape: (12, 12)
> Open shape: (48, 48)


## Hermiticity is a required acceptance check

In [4]:
for name, matrix in {"bulk": h_bulk, "ribbon": h_ribbon, "open": h_open}.items():
    print(f"{name:>6}: max |H - H†| = {hermiticity_residual(matrix):.3e}")

  bulk: max |H - H†| = 0.000e+00
ribbon: max |H - H†| = 0.000e+00
  open: max |H - H†| = 0.000e+00


## Analytic derivatives prepared for the Kubo formula

We validate the derivatives numerically now, but do not yet evaluate Berry curvature.

In [5]:
kx, ky, step = 0.12, -0.31, 1.0e-7
d_kx, d_ky = bloch_hamiltonian_derivatives(kx, ky, params)
fd_kx = (bloch_hamiltonian(kx + step, ky, params) - bloch_hamiltonian(kx - step, ky, params)) / (2 * step)
fd_ky = (bloch_hamiltonian(kx, ky + step, params) - bloch_hamiltonian(kx, ky - step, params)) / (2 * step)

print("max error dH/dkx:", np.max(np.abs(d_kx - fd_kx)))
print("max error dH/dky:", np.max(np.abs(d_ky - fd_ky)))

max error dH/dkx: 2.120837341443175e-10
max error dH/dky: 3.2324309895415126e-09


---